# DEST — Anexo #8: Halton vs Sobol vs V3 (STANDALONE)

**No subas nada — Ejecuta todo**

* **Pregunta:** ¿Collatz es especial o cualquier QMC híbrida funciona? Halton es otra secuencia QMC (van der Corput base 2) distinta de Sobol.
* **Dataset:** CIFAR-10 → ResNet9, 15 épocas, batch 128, SGD 0.01 cosine (idéntico a PAPER)
* **Samplers:** `stochastic` (control) / `sobol` (QMC puro) / `halton` (QMC puro) / `collatz_v3` (QMC+annealing 0→0.5)
* **Diseño:** 10 seeds 200–209 pareadas (cada seed = cuarteto completo) → 40 runs, ~80 min T4, reanudable

**Entregable:** Tabla Sobol vs Halton vs V3 — si Halton≈Sobol≈stochastic pero V3 gana, Collatz+annealing es especial; si Halton≈V3, es efecto QMC+annealing genérico.


In [ ]:
# 0. Setup standalone — clona repo si hace falta
import os, sys, subprocess, shutil, glob
print("🔧 Setup...")
has = os.path.exists("DEST/src/dest")
if not os.path.exists("dest_lib") and not has:
    print("Clonando https://github.com/starlyn2010/DEST ...")
    subprocess.check_call(["git","clone","https://github.com/starlyn2010/DEST.git"])
    has = os.path.exists("DEST/src/dest")
if os.path.exists("DEST/src/dest"):
    print("Instalando DEST...")
    subprocess.check_call([sys.executable,"-m","pip","install","-e","DEST","-q"])
    if "DEST/src" not in sys.path: sys.path.insert(0,"DEST/src")
    import dest
    sys.modules["dest_lib"]=dest
    for sub in ["config","samplers","models","datasets","runner","metrics","reproducibility"]:
        try:
            m=__import__(f"dest.{sub}", fromlist=[sub]); sys.modules[f"dest_lib.{sub}"]=m
        except: pass
    print("✅ alias dest_lib->dest")
# verificar Halton
from dest_lib.samplers import HaltonPermutationSampler, SamplerFactory
print("✅ HaltonSampler:", HaltonPermutationSampler)
try:
    import scipy; print("scipy",scipy.__version__)
except: subprocess.check_call([sys.executable,"-m","pip","install","scipy","-q"])
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 1. Config Anexo #8
from dest_lib.config import get_config
config=get_config("PAPER")
config["datasets"]=["CIFAR10"]
config["samplers"]=["stochastic","sobol","halton","collatz_v3"]
config["seeds"]=list(range(200,210))
config["epochs"]=15
config["batch_size"]=128
config["lr"]=0.01
config["lr_schedule"]="cosine"
config["output_dir"]="./dest_halton_anexo8"
config["val_fraction"]=0.1
config["verbose"]=True
import json
print(json.dumps({k:config[k] for k in ["datasets","samplers","seeds","epochs","batch_size","lr","output_dir"]},indent=2))
print(f"Total runs: {len(config['seeds'])*len(config['samplers'])} (40)")
print(f"Tiempo estimado: ~{40*3.5:.0f} min T4")

In [ ]:
# 2. Ejecutar reanudable (cuarteto por seed)
import os, time, json, dataclasses
from dest_lib.runner import ExperimentRunner
runner=ExperimentRunner(config)
total=len(config["seeds"])*len(config["samplers"])
done=0
start_all=time.time()
for seed in config["seeds"]:
    for sampler_name in config["samplers"]:
        exp_id=f"CIFAR10_{sampler_name}"
        out_file=os.path.join(config["output_dir"], f"{exp_id}_{sampler_name}_seed_{seed}.json")
        if os.path.exists(out_file):
            print(f"⏭️ Saltando {exp_id} seed {seed}")
            done+=1
            continue
        print(f"\n[{done+1}/{total}] {exp_id} seed {seed}")
        try:
            r=runner.run_single_seed(exp_id=exp_id, sampler_name=sampler_name, seed=seed, dataset="CIFAR10")
            print(f"  ✅ {sampler_name} seed {seed}: {r.final_test_acc:.2f}% en {r.total_runtime_seconds/60:.1f} min")
        except Exception as e:
            print(f"  ❌ {e}"); import traceback; traceback.print_exc()
        done+=1
print(f"\n✅ Anexo #8 completo {done}/{total} en {(time.time()-start_all)/60:.1f} min")

In [ ]:
# 3. Resumen estadístico
import glob, json, numpy as np, os
from collections import defaultdict
pattern=os.path.join(config["output_dir"],"*.json")
files=glob.glob(pattern)
print(f"JSONs: {len(files)}")
if files:
    groups=defaultdict(list)
    for f in files:
        j=json.load(open(f)); groups[j["sampler_name"]].append(j)
    for s in config["samplers"]:
        arr=[j["final_test_acc"] for j in groups[s]]
        if arr: print(f"{s:12s}: {np.mean(arr):.2f} ±{np.std(arr,ddof=1):.2f} n={len(arr)} [{min(arr):.2f}-{max(arr):.2f}]")
    # pareado V3 vs cada uno
    if "collatz_v3" in groups:
        for other in ["stochastic","sobol","halton"]:
            if other not in groups: continue
            v3={j["seed"]:j["final_test_acc"] for j in groups["collatz_v3"]}
            ot={j["seed"]:j["final_test_acc"] for j in groups[other]}
            common=sorted(set(v3)&set(ot))
            diffs=[v3[s]-ot[s] for s in common]
            if diffs:
                from scipy import stats
                t,p=stats.ttest_rel([v3[s] for s in common],[ot[s] for s in common])
                d=np.mean(diffs)/np.std(diffs,ddof=1) if np.std(diffs,ddof=1)>0 else 0
                print(f"\nV3 vs {other} (n={len(common)}): diff {np.mean(diffs):+.2f} t={t:.2f} p={p:.4f} d={d:.2f} gana {sum(1 for x in diffs if x>0)}/{len(diffs)}")
    # curvas
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,4))
    for s in config["samplers"]:
        if s not in groups: continue
        arr=np.array([j["test_accs"] for j in groups[s]])
        m=arr.mean(0); sd=arr.std(0,ddof=1)
        plt.plot(range(1,config["epochs"]+1), m, label=s)
        plt.fill_between(range(1,config["epochs"]+1), m-sd, m+sd, alpha=0.15)
    plt.xlabel("Época"); plt.ylabel("Test acc %"); plt.title("CIFAR-10 Halton vs Sobol vs V3")
    plt.legend(); plt.grid(alpha=0.3)
    plt.savefig(os.path.join(config["output_dir"],"cifar10_halton_curvas.png"),dpi=200,bbox_inches="tight")
    plt.show()

In [ ]:
# 4. Zip y descarga
import os, shutil
out_dir=config["output_dir"]
zip_name="resultados_Halton_Anexo8.zip"
if os.path.exists(out_dir) and len(os.listdir(out_dir))>0:
    shutil.make_archive(zip_name.replace(".zip",""),'zip',out_dir)
    print(f"✅ {zip_name} {os.path.getsize(zip_name)/1e6:.2f} MB, {len(os.listdir(out_dir))} archivos")
    try:
        from google.colab import files; files.download(zip_name)
    except: print("No Colab —",os.path.abspath(zip_name))
else: print("⚠️ nada que empaquetar")

**Siguiente:** copiar zip a `dest/dest_results_paper/` local → re-ejecutar `anexos_analysis/analyze_jsons.py` → párrafo blog: *“Halton [X] vs Sobol [Y] vs V3 [Z] — QMC puro no basta, annealing es clave”* o viceversa. Luego #2 barrido alpha.